# Colon dataset selection

Nowy narząd, ta sama metodyka co `brain_dataset.ipynb`: `msi_dataset_manager.exploration.DatasetExplorer` + `DatasetReview`/`DatasetReviewProfile` (`packages/msi_dataset_manager/src/msi_dataset_manager/exploration/dataset_review.py`) do obiektywnej detekcji duplikatów, wariantów QC i heurystyki morfologii. Zero zmian w bibliotece, zero pobrań surowych danych — wynikiem jest wyłącznie przejrzana lista kandydatów wyeksportowana jako `filter.json`/`selection.json`.

**Uwaga na wielkość puli:** 17 kandydatów przy `Mouse`+`Negative` (surowe METASPACE: 237, wszystkie organizmy/polaryzacje). Prawie wszystkie pochodzą z jednej serii laboratoryjnej `MPIMM_5xx`–`MPIMM_6xx` (Oct 2025–Mar 2026) — to jedno spójne badanie z wieloma zwierzętami (`M1`…`M10`, grupy `ctrl4W`/`PEM4W`), nie 17 niezależnych projektów.

**Zakres m/z: `mz_min=200, mz_max=900`, przyjęty jako domyślny punkt startowy spójny z kidney** (patrz `brain_dataset.ipynb`, sekcja 5, gdzie ten wariant wypadł najlepiej na dostępnym katalogu). To NIE jest wynik osobnej optymalizacji dla tego narządu — ujednolicenie zakresu m/z między wszystkimi narządami jest świadomie odłożone na później (tak jak w `kidney_dataset_repaired.ipynb`/`liver_dataset_repaired.ipynb`). Poniżej pokazuję explicité, ile kandydatów ten zakres realnie pokrywa, żebyś widział koszt tego wyboru dla tego konkretnego narządu.

In [1]:
import os
from pathlib import Path

current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").is_file()
)
os.chdir(repository_root)

repository_root

PosixPath('/home/max/repositories/MSIAutoEncoderWrapper')

In [2]:
import re

import pandas as pd
from IPython.display import display

from msi_dataset_manager.exploration import DatasetExplorer, DatasetReviewProfile

# REMARK: date i download DB is 12.08.2026 (DD, MM, YYYY) -- same cache as the other notebooks.
explorer = DatasetExplorer(
    source="metaspace",
    cache_dir="assets/local/datasets/metaspace",
    refresh_cache=False,
)

## 1. Szeroka pula kandydatów

Ten sam filtr biologiczny co `brain_dataset.ipynb`: `condition=["Wildtype", "Wtype", "N/A"]`, `organism=Mouse`, `polarity=Negative`. Bez `mz_min`/`mz_max` na tym etapie, żeby audyt duplikatów/jakości objął całą pulę, nie tylko to, co już pasuje do docelowego zakresu.

In [3]:
broad_filters = {
    "organism": "Mouse",
    "organism_part": "Colon",
    "condition": ["Wildtype", "Wtype", "N/A"],
    "polarity": "Negative",
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
}
results = explorer.filter(broad_filters)
print(f"Found {len(results)} datasets")
display(results[["dataset_id", "name", "condition", "analyzer_type", "ionisation_source", "mz_min", "mz_max", "pixel_count"]])

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

Found 17 datasets


,dataset_id,name,condition,analyzer_type,ionisation_source,mz_min,mz_max,pixel_count
0,2026-03-06_09h01m03s,MPIMM_627_biopsy18_0,N/A,Orbitrap,AP-SMALDI5 AF,399.993988,1199.986328,105376
1,2026-03-06_09h02m07s,MPIMM_628_biopsy18_2,N/A,Orbitrap,AP-SMALDI5 AF,399.993958,1199.984619,64000
2,2026-03-03_10h00m04s,MPIMM_625_M4_colon_PEM4W,N/A,Orbitrap,AP-SMALDI5 AF,199.997116,999.986938,121888
3,2026-03-03_10h00m48s,MPIMM_626_M8_colon_PEM4W,N/A,Orbitrap,AP-SMALDI5 AF,199.997208,999.986084,93835
4,2026-03-03_09h48m49s,MPIMM_624_M9_colon_ctrl4W,N/A,Orbitrap,AP-SMALDI5 AF,199.998520,999.987061,104244
5,2026-02-20_09h10m14s,MPIMM_621_M10_colon_ctrl4W,N/A,Orbitrap,AP-SMALDI5 AF,199.999176,999.989258,121590
6,2026-02-18_08h55m08s,MPIMM_619_M1_colon_PEM4W,N/A,Orbitrap,AP-SMALDI5 AF,199.999176,999.990051,92112
7,2025-12-17_08h29m01s,MPIMM_605_M2_colon_PEM4W,N/A,Orbitrap,AP-SMALDI5 AF,199.998398,999.987671,114019
8,2025-12-01_09h03m54s,MPIMM_603,N/A,Orbitrap,AP-SMALDI5 AF,199.998322,999.992126,141246
9,2025-11-27_15h26m51s,MPIMM_602,N/A,Orbitrap,AP-SMALDI5 AF,99.999405,399.995361,30318


## 2. Ręczna kontrola jakości, której żadna reguła biblioteczna nie łapie

Ten sam skan co w `kidney_dataset_repaired.ipynb`: niedopasowanie gatunku/tkanki w nazwie, jawne oznaczenia testowe.

In [4]:
suspect_pattern = r"zebrafish|drosophila|\brat\b|\bhuman\b|\btest\b|\(test\)|calib|standard"
suspects = results[results["name"].str.contains(suspect_pattern, case=False, na=False, regex=True)]
display(suspects[["dataset_id", "name", "condition", "organisms", "pixel_count"]] if len(suspects) else "none found")

'none found'

## 3. Przegląd biblioteczny (`DatasetExplorer.review_current`)

Brak wbudowanego profilu `"colon"` w `_PROFILES` (tylko `brain`/`liver`) — przekazuję `DatasetReviewProfile` z poziomu notebooka, tak jak w `kidney_dataset_repaired.ipynb`.

In [5]:
colon_profile = DatasetReviewProfile(
    low_pixel_threshold=None,  # minimum w tej puli to 30 318 pikseli -- nie ma outlierów do flagowania.
    morphology_pattern=r"(?:crypt|mucosa|submucosa|serosa|muscularis|epithel)",
    explicit_regional_names=frozenset(),
)
review = explorer.review_current(profile=colon_profile)

print("available rules:", review.available_rules)
display(review.summary())

display(
    review.table.loc[
        review.table["duplicate_cluster_size"] > 1,
        ["duplicate_cluster_id", "dataset_id", "name", "pixel_count", "duplicate_confidence", "duplicate_excluded", "recommended_keeper_dataset_id"],
    ].sort_values(["duplicate_confidence", "duplicate_cluster_id"])
)
display(
    review.table.loc[
        review.table["mz_shift_qc_variant"] | review.table["morphology_hint"].eq("regional_or_microregion") | review.table["low_pixel_flag"],
        ["dataset_id", "name", "pixel_count", "mz_shift_qc_variant", "morphology_hint", "low_pixel_flag"],
    ]
)

available rules: ('high_confidence_duplicates', 'mz_shift_qc_variants', 'explicit_regional_fragments')


,rule,dataset_count
0,high_confidence_duplicates,0
1,mz_shift_qc_variants,0
2,explicit_regional_fragments,0


,duplicate_cluster_id,dataset_id,name,pixel_count,duplicate_confidence,duplicate_excluded,recommended_keeper_dataset_id


,dataset_id,name,pixel_count,mz_shift_qc_variant,morphology_hint,low_pixel_flag


## 4. Zastosowanie reguł i finalna lista

`high_confidence_duplicates` + `mz_shift_qc_variants` + `explicit_regional_fragments` (wszystkie trzy, dla spójności z pozostałymi notebookami, nawet gdy akurat wychodzi 0). `morphology_hint`/`low_pixel_flag` zostają doradczo.

In [6]:
applied_rules = ["high_confidence_duplicates", "mz_shift_qc_variants", "explicit_regional_fragments"]
explorer.apply_review(review, rules=applied_rules)

final_filters = {
    "organism": "Mouse",
    "organism_part": "Colon",
    "condition": ["Wildtype", "Wtype", "N/A"],
    "polarity": "Negative",
    "mz_min": 200,
    "mz_max": 900,
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
    "include_molecule_stats": True,
    "include_spatial_annotation_stats": False,  # see brain_dataset.ipynb section 6 for the cost rationale
}
# exclude_dataset_ids intentionally omitted -- session-level exclusions from steps 2/3/4 persist across this re-query.
results_colon = explorer.filter(final_filters)
print(f"final colon shortlist: {len(results_colon)} datasets (of {len(results)} broad candidates)")
display(results_colon[["dataset_id", "name", "analyzer_type", "pixel_count", "molecule_count", "unique_molecule_count"]])

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

final colon shortlist: 7 datasets (of 17 broad candidates)


,dataset_id,name,analyzer_type,pixel_count,molecule_count,unique_molecule_count
0,2026-03-03_10h00m04s,MPIMM_625_M4_colon_PEM4W,Orbitrap,121888,170,8
1,2026-03-03_10h00m48s,MPIMM_626_M8_colon_PEM4W,Orbitrap,93835,195,7
2,2026-03-03_09h48m49s,MPIMM_624_M9_colon_ctrl4W,Orbitrap,104244,236,16
3,2026-02-20_09h10m14s,MPIMM_621_M10_colon_ctrl4W,Orbitrap,121590,174,1
4,2026-02-18_08h55m08s,MPIMM_619_M1_colon_PEM4W,Orbitrap,92112,215,9
5,2025-12-17_08h29m01s,MPIMM_605_M2_colon_PEM4W,Orbitrap,114019,204,13
6,2025-12-01_09h03m54s,MPIMM_603,Orbitrap,141246,286,42


## 5. Grupowanie w serie biologiczne (Poziom 3)

Ta sama heurystyka co w `brain_dataset.ipynb` — tylko do identyfikacji grup, które muszą zostać razem w tym samym podziale train/validation/test, nie do automatycznego wybierania reprezentanta.

In [7]:
def biological_series_key(name: str) -> str:
    s = str(name).lower()
    s = re.sub(r"^\d{4}-\d{2}-\d{2}[_ ]", "", s)
    s = re.sub(r"^\d{8}_+", "", s)
    s = re.sub(r"_(?:aq_ml|aq|ml)$", "", s)
    s = re.sub(r"_\d+ppm$", "", s)
    s = re.sub(r"-total ion count$", "", s)
    s = re.sub(r" - root mean square$", "", s)
    s = re.sub(r"_replicate\d+$", "", s)
    s = re.sub(r"_s\d+$", "", s)
    s = re.sub(r"_\d+$", "", s)
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    return s


results_colon["biological_series_id"] = results_colon["name"].apply(biological_series_key)
n_series = results_colon["biological_series_id"].nunique()
print(f"final shortlist: {len(results_colon)} datasets across {n_series} name-derived series")
series_sizes = results_colon.groupby("biological_series_id").size().sort_values(ascending=False)
display(series_sizes[series_sizes > 1])

final shortlist: 7 datasets across 7 name-derived series


Series([], dtype: int64)

In [8]:
output_path = Path("data/colon_workspace/configs/datasets/colon")
exported = explorer.export_selection(output_path, sort_by="download_size_bytes", ascending=False)
exported

{'filters': PosixPath('data/colon_workspace/configs/datasets/colon/filter.json'),
 'selection': PosixPath('data/colon_workspace/configs/datasets/colon/selection.json')}

## Podsumowanie

- Szeroka pula (`Mouse`+`Negative`+`Wildtype`/`Wtype`/`N/A`): patrz sekcja 1 dla dokładnej liczby.
- Kontrola jakości i przegląd biblioteczny: sekcje 2–3.
- Zakres m/z `200–900` przyjęty jako domyślny, spójny z kidney — **nie zoptymalizowany osobno dla tego narządu**, patrz zastrzeżenie na początku notebooka.
- Eksport do `data/colon_workspace/configs/datasets/colon/` — pierwszy raz dla tego narządu, brak wcześniejszej konfiguracji do porównania.
- Analiza wspólnego zakresu m/z między wszystkimi narządami — świadomie pominięta, do rozwiązania osobno.